In [1]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'
REPO_DIR = Path('/content/Yolo-ST-GCN')
BRANCH = 'duy'

# Install lightweight dependencies needed for smoke checks.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'numpy', 'scipy', 'torch', '-q'],
    check=True,
)

# Clone or update exact branch used for current development.
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready:', os.getcwd())
print('Branch:', BRANCH)

Repo ready: /content/Yolo-ST-GCN
Branch: duy


# ST-GCN Smoke Test
Notebook này chạy smoke test nhanh cho pipeline sau khi tách Penn/COCO dataset.

Mục tiêu:
- Xác nhận import module thành công
- Xác nhận forward pass của model
- Xác nhận build tensors cho Penn và COCO synthetic đều trả về định dạng chung `(N, 2, T, 14, 1)`

In [2]:
import os
import sys
import subprocess
from pathlib import Path


def find_repo_root() -> Path | None:
    """Try common Colab/local locations for this repo."""
    candidates = [Path.cwd(), Path('/content'), Path('/content/drive/MyDrive')]

    for base in candidates:
        if (base / 'src').exists() and (base / 'scripts').exists():
            return base

    markers = ['Yolo-ST-GCN', 'ST-GCN']
    for base in candidates:
        if not base.exists():
            continue
        for m in markers:
            found = list(base.glob(f'**/{m}'))[:5]
            for p in found:
                if (p / 'src').exists() and (p / 'scripts').exists():
                    return p
    return None


def ensure_repo_available() -> Path | None:
    repo = find_repo_root()
    if repo is not None:
        return repo

    # Colab fallback: clone repo if missing.
    colab_repo = Path('/content/Yolo-ST-GCN')
    if not colab_repo.exists():
        print('Repo not found in kernel filesystem. Cloning from GitHub...')
        subprocess.run(
            ['git', 'clone', 'https://github.com/schizocatto/Yolo-ST-GCN.git', str(colab_repo)],
            check=True,
        )
    if (colab_repo / 'src').exists() and (colab_repo / 'scripts').exists():
        return colab_repo

    return None


repo_root = ensure_repo_available()
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Python:', sys.version.split()[0])
print('CWD:', Path.cwd())
print('Repo root detected:', repo_root)
print('src import ready:', repo_root is not None and (repo_root / 'src').exists())

Python: 3.12.13
CWD: /content/Yolo-ST-GCN
Repo root detected: /content/Yolo-ST-GCN
src import ready: True


In [3]:
import inspect
import os
import tempfile
import numpy as np
import scipy.io
import torch

from src.config import TARGET_FRAMES, EXERCISE_CLASSES
from src.model import Model_STGCN
from src.dataset import build_data_tensors

has_dataset_format = 'dataset_format' in inspect.signature(build_data_tensors).parameters
print('build_data_tensors supports dataset_format:', has_dataset_format)

# 1) Model forward smoke
x = np.random.randn(2, 2, TARGET_FRAMES, 14, 1).astype(np.float32)
model = Model_STGCN(num_classes=len(EXERCISE_CLASSES))
out = model(torch.from_numpy(x))
print('Model forward output shape:', tuple(out.shape))

# 2) Penn synthetic dataset smoke
with tempfile.TemporaryDirectory(prefix='penn_smoke_') as td:
    action = EXERCISE_CLASSES[0]
    n_frames = 48
    mat = {
        'x': np.random.randint(0, 640, (n_frames, 13)).astype(float),
        'y': np.random.randint(0, 480, (n_frames, 13)).astype(float),
        'action': np.array([[action]]),
        'nframes': np.array([[n_frames]]),
        'train': np.array([[1]]),
    }
    scipy.io.savemat(os.path.join(td, '0001.mat'), mat)
    if has_dataset_format:
        data_p, labels_p, flags_p, _, vids_p = build_data_tensors(td, dataset_format='penn')
    else:
        data_p, labels_p, flags_p, _, vids_p = build_data_tensors(td)
    print('Penn tensor shape:', data_p.shape)
    print('Penn labels:', labels_p.tolist(), 'flags:', flags_p.tolist(), 'video_ids:', vids_p)

assert out.shape[0] == 2 and out.shape[1] == len(EXERCISE_CLASSES)
assert data_p.shape[1:] == (2, TARGET_FRAMES, 14, 1)

# 3) COCO synthetic dataset smoke (only when new API is available)
if has_dataset_format:
    with tempfile.TemporaryDirectory(prefix='coco_smoke_') as td:
        action = EXERCISE_CLASSES[1]
        kpts = np.random.rand(40, 17, 2).astype(np.float32) * np.array([640.0, 480.0], dtype=np.float32)
        np.savez(
            os.path.join(td, 'sample_0001.npz'),
            keypoints=kpts,
            action=np.array([action]),
            train=np.array([1]),
            video_id=np.array(['sample_0001'])
        )
        data_c, labels_c, flags_c, _, vids_c = build_data_tensors(td, dataset_format='coco')
        print('COCO->Penn tensor shape:', data_c.shape)
        print('COCO labels:', labels_c.tolist(), 'flags:', flags_c.tolist(), 'video_ids:', vids_c)
        assert data_c.shape[1:] == (2, TARGET_FRAMES, 14, 1)
    print('SMOKE TEST: PASS (Penn + COCO)')
else:
    print('SMOKE TEST: PASS (Penn only)')
    print('Note: Repo branch on GitHub chua co API dataset_format; can push latest code de test COCO.')

build_data_tensors supports dataset_format: True
Model forward output shape: (2, 8)
[dataset] No official test split found for Penn data; using last 30% per class as test.
Penn tensor shape: (1, 2, 64, 14, 1)
Penn labels: [0] flags: [0] video_ids: ['0001']
[dataset] No test split found for COCO data; using last 30% per class as test.
COCO->Penn tensor shape: (1, 2, 64, 14, 1)
COCO labels: [1] flags: [0] video_ids: ['sample_0001']
SMOKE TEST: PASS (Penn + COCO)


## Train + Inference on Gym288-skeleton
Chạy 2 cell bên dưới để huấn luyện và đánh giá ST-GCN trên Gym288-skeleton.

Yêu cầu dữ liệu:
- File pickle theo format docs: `gym288_skeleton.pkl`
- Có keys `split.train`, `split.test`, `annotations`

Kết quả sẽ được lưu trong `outputs/gym288/`.

In [4]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

# Thu muc luu dataset Gym288-skeleton trong Colab
DOWNLOAD_DIR = Path('/content/Gym288-skeleton')

if DOWNLOAD_DIR.exists() and any(DOWNLOAD_DIR.iterdir()):
    print('Dataset da ton tai — bo qua buoc tai xuong.')
else:
    print('Dang tai dataset Gym288-skeleton tu Hugging Face...')
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='Lozumi/Gym288-skeleton',
        repo_type='dataset',
        local_dir=str(DOWNLOAD_DIR),
        local_dir_use_symlinks=False,
    )
    print('Da tai xong!')

print('Cau truc thu muc tai ve:')
for item in sorted(os.listdir(DOWNLOAD_DIR)):
    print(f' - {item}')

# Tu dong tim file pickle dau vao cho train/inference
pkl_candidates = sorted(DOWNLOAD_DIR.rglob('*.pkl'))
if not pkl_candidates:
    raise FileNotFoundError('Khong tim thay file .pkl trong Gym288-skeleton sau khi tai.')

GYM288_DATASET_PATH = pkl_candidates[0]
OUT_DIR = Path('outputs/gym288')

print('Using dataset pickle:', GYM288_DATASET_PATH)
print('Output dir:', OUT_DIR)

Dang tai dataset Gym288-skeleton tu Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Da tai xong!
Cau truc thu muc tai ve:
 - .cache
 - .gitattributes
 - README.md
 - gym_288_skeleton.pkl
Using dataset pickle: /content/Gym288-skeleton/gym_288_skeleton.pkl
Output dir: outputs/gym288


In [8]:
import pickle
import re
import urllib.request
from pathlib import Path

import torch

from src.gym99_dataset import build_gym99_data_tensors
from src.model import Model_STGCN

if 'GYM288_DATASET_PATH' not in globals():
    raise RuntimeError('Hay chay cell tai Gym288 truoc de tao bien GYM288_DATASET_PATH.')

print('Dang tai danh sach class tu server FineGym...')
url_288 = 'https://sdolivia.github.io/FineGym/resources/dataset/gym288_categories.txt'
url_99 = 'https://sdolivia.github.io/FineGym/resources/dataset/gym99_categories.txt'

text_288 = urllib.request.urlopen(url_288).read().decode('utf-8')
text_99 = urllib.request.urlopen(url_99).read().decode('utf-8')

# Parse robustly from FineGym format:
# "Clabel: <id>; set: <id>; Glabel: <id>; ..."
pattern = re.compile(r'Clabel:\s*(\d+)\s*;\s*set:\s*\d+\s*;\s*Glabel:\s*(\d+)\s*;', re.MULTILINE)

pairs_288 = [(int(c), int(g)) for c, g in pattern.findall(text_288)]
pairs_99 = [(int(c), int(g)) for c, g in pattern.findall(text_99)]

if not pairs_288 or not pairs_99:
    raise RuntimeError('Khong parse duoc Clabel/Glabel tu file category FineGym.')

clabel288_to_glabel = dict(pairs_288)            # 288 clabel -> global glabel
glabel_to_clabel99 = {g: c for c, g in pairs_99} # global glabel -> 99 clabel

# Final map: Gym288 clabel -> Gym99 clabel via shared Glabel
map_288_to_99 = {}
for clabel_288, glabel in clabel288_to_glabel.items():
    if glabel in glabel_to_clabel99:
        map_288_to_99[clabel_288] = glabel_to_clabel99[glabel]

print(f'Thanh cong! Da tim thay anh xa cho {len(map_288_to_99)} lop Gym99.')

# Build Gym99-from-Gym288 pickle by relabeling annotations
with open(str(GYM288_DATASET_PATH), 'rb') as f:
    gym288_payload = pickle.load(f)

annotations = gym288_payload.get('annotations', [])
split_info = gym288_payload.get('split', {})

gym99_annotations = []
valid_ids = set()
for ann in annotations:
    lbl_288 = int(ann['label'])
    if lbl_288 not in map_288_to_99:
        continue

    ann_new = dict(ann)
    ann_new['label'] = int(map_288_to_99[lbl_288])
    gym99_annotations.append(ann_new)
    valid_ids.add(str(ann_new['frame_dir']))

gym99_split = {
    'train': [vid for vid in split_info.get('train', []) if vid in valid_ids],
    'test': [vid for vid in split_info.get('test', []) if vid in valid_ids],
}

gym99_payload = {
    'split': gym99_split,
    'annotations': gym99_annotations,
}

GYM99_DIR = Path('/content/Gym99-from-Gym288')
GYM99_DIR.mkdir(parents=True, exist_ok=True)
GYM99_DATASET_PATH = GYM99_DIR / 'gym99_from_gym288.pkl'
with open(str(GYM99_DATASET_PATH), 'wb') as f:
    pickle.dump(gym99_payload, f)

print('Gym99 pickle generated:', GYM99_DATASET_PATH)
print('Gym99 annotations:', len(gym99_annotations))
print('Gym99 split train/test:', len(gym99_split['train']), len(gym99_split['test']))

# Quick test on a few Gym99 test samples
sample_count = 8
data, labels, flags, raw_counts, video_ids = build_gym99_data_tensors(
    dataset_path=str(GYM99_DATASET_PATH),
    split='test',
    keep_unknown_split=False,
)

print('Loaded Gym99 test samples:', len(data))
print('Tensor shape:', data.shape)
print('Label shape:', labels.shape)

n = min(sample_count, len(data))
print(f'Preview first {n} samples:')
for i in range(n):
    print(f'  {i:02d} | video_id={video_ids[i]} | label={int(labels[i])} | raw_frames={raw_counts[i]}')

if n > 0:
    x = torch.from_numpy(data[:n]).float()
    model = Model_STGCN(num_classes=99).eval()
    with torch.no_grad():
        logits = model(x)
        pred = torch.argmax(logits, dim=1).cpu().numpy()
    print('Forward pass logits shape:', tuple(logits.shape))
    print('Predictions (untrained model, sanity check only):', pred.tolist())
else:
    print('No test sample available to run forward pass.')

Dang tai danh sach class tu server FineGym...
Thanh cong! Da tim thay anh xa cho 99 lop Gym99.
Gym99 pickle generated: /content/Gym99-from-Gym288/gym99_from_gym288.pkl
Gym99 annotations: 34240
Gym99 split train/test: 25826 8414
Loaded Gym99 test samples: 8414
Tensor shape: (8414, 2, 64, 14, 1)
Label shape: (8414,)
Preview first 8 samples:
  00 | video_id=A0xAXXysHUo_002184_002237_0035_0036 | label=93 | raw_frames=48
  01 | video_id=AZ4wWG6Rcak_003643_003675_0024_0026 | label=77 | raw_frames=61
  02 | video_id=AZ4wWG6Rcak_005608_005706_0045_0046 | label=49 | raw_frames=23
  03 | video_id=AZ4wWG6Rcak_006115_006196_0078_0080 | label=72 | raw_frames=53
  04 | video_id=AZ4wWG6Rcak_006218_006294_0021_0021 | label=67 | raw_frames=23
  05 | video_id=BLzs5opw8uM_001044_001143_0048_0050 | label=59 | raw_frames=36
  06 | video_id=BLzs5opw8uM_001671_001766_0030_0030 | label=60 | raw_frames=29
  07 | video_id=BLzs5opw8uM_002316_002397_0049_0051 | label=62 | raw_frames=47
Forward pass logits shape: 

In [ ]:
import os
import pickle
import select
import subprocess
import sys
import time
import numpy as np
from pathlib import Path

# Retrain probe: 2-stream + joint format report
if 'GYM99_DATASET_PATH' not in globals():
    raise RuntimeError('Hay chay cell tao Gym99 truoc de co GYM99_DATASET_PATH.')

print('=== Joint Format Check ===')
with open(str(GYM99_DATASET_PATH), 'rb') as f:
    payload = pickle.load(f)

annotations = payload.get('annotations', [])
if not annotations:
    raise RuntimeError('Gym99 pickle khong co annotations.')

sample = annotations[0]
if 'keypoint' in sample:
    kp = np.asarray(sample['keypoint'])
    print('Raw keypoint shape in pickle:', kp.shape)
    if kp.ndim == 4 and kp.shape[-2] == 17:
        print('Co 17 khop COCO trong du lieu goc: YES')
    else:
        print('Co 17 khop COCO trong du lieu goc: UNKNOWN')
else:
    print("Khong thay truong 'keypoint' trong annotation mau.")

print('Khop ao thu 18 co san trong pickle: NO')
print('Trang thai hien tai pipeline: KHONG train truc tiep 17+1; loader hien remap ve 14 khop (13 Penn + 1 virtual).')
print('=> 2-stream hien tai dang train tren 14 khop da tinh bone stream.')

print('\n=== Start 2-Stream Retrain (quick run) ===')
out_dir = 'outputs/gym99_2s_probe'
script_path = Path('scripts/train_gym99.py')
print('CWD:', Path.cwd())
print('Script exists:', script_path.exists(), script_path)

train_cmd = [
    sys.executable, '-u', 'scripts/train_gym99.py',
    '--dataset_path', str(GYM99_DATASET_PATH),
    '--out_dir', out_dir,
    '--epochs', '1',
    '--batch_size', '256',
    '--lr', '0.001',
    '--num_wokers', '2',
    '--use_two_stream',
]
print('Running:', ' '.join(train_cmd))

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

# Stream stdout/stderr live and print heartbeat if tqdm does carriage-return only.
proc = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

saw_output = False
last_msg = time.time()
assert proc.stdout is not None

while True:
    ready, _, _ = select.select([proc.stdout], [], [], 5)
    if ready:
        line = proc.stdout.readline()
        if line == '' and proc.poll() is not None:
            break
        if line:
            saw_output = True
            print(line, end='')
            last_msg = time.time()
    else:
        if proc.poll() is None:
            elapsed = int(time.time() - last_msg)
            print(f'[train] still running... ({elapsed}s since last newline log)')
        else:
            break

for tail in proc.stdout:
    saw_output = True
    print(tail, end='')

ret = proc.wait()
if not saw_output:
    print('Khong nhan duoc log train tu subprocess (stdout rong).')
if ret != 0:
    raise subprocess.CalledProcessError(ret, train_cmd)

print('\nProbe train done. Output dir:', out_dir)
print('- weights: outputs/gym99_2s_probe/stgcn_gym99_2s.pth')
print('- metrics: outputs/gym99_2s_probe/metrics_train_gym99.json')

=== Joint Format Check ===
Raw keypoint shape in pickle: (1, 48, 17, 2)
Co 17 khop COCO trong du lieu goc: YES
Khop ao thu 18 co san trong pickle: NO
Trang thai hien tai pipeline: KHONG train truc tiep 17+1; loader hien remap ve 14 khop (13 Penn + 1 virtual).
=> 2-stream hien tai dang train tren 14 khop da tinh bone stream.

=== Start 2-Stream Retrain (quick run) ===
CWD: /content/Yolo-ST-GCN
Script exists: True scripts/train_gym99.py
Running: /usr/bin/python3 -u scripts/train_gym99.py --dataset_path /content/Gym99-from-Gym288/gym99_from_gym288.pkl --out_dir outputs/gym99_2s_probe --epochs 1 --batch_size 256 --lr 0.001 --num_wokers 2 --use_two_stream
[train] still running... (5s since last newline log)
Device: cpu
[train] still running... (5s since last newline log)
[train] still running... (10s since last newline log)
[train] still running... (15s since last newline log)
Loading Gym99-skeleton dataset...
Loaded 34240 samples  train=25826  test=8414
num_classes=99 (inferred=99)
DataLoa

In [ ]:
# !git pull

In [ ]:
# !python3 scripts/train_gym288.py \
# --dataset_path /content/Gym288-skeleton/gym_288_skeleton.pkl \
# --out_dir outputs/gym288 \
# --epochs 30 \
# --batch_size 512 \
# --lr 0.001 \
# --num_wokers 2 \
# --use_two_stream

In [6]:

# # 2) Inference/Evaluation on Gym288 test split
# weights_path = OUT_DIR / 'stgcn_gym288.pth'
# infer_cmd = [
#     sys.executable, 'scripts/inference_gym288.py',
#     '--dataset_path', str(GYM288_DATASET_PATH),
#     '--weights', str(weights_path),
#     '--out_dir', str(OUT_DIR),
#     '--batch_size', '64',
#     '--topk', '5',
# ]
# print('Running inference command:\n', ' '.join(infer_cmd))
# subprocess.run(infer_cmd, check=True)

# print('\nDone. Check outputs in:', OUT_DIR)
# print('- metrics_train_gym288.json')
# print('- metrics_inference_gym288.json')
# print('- predictions_test_top1.json')